In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3

from scipy.stats import norm

In [4]:
conn = sqlite3.connect("../bluestock_mf.db")

nav_df = pd.read_sql("select * from nav_history",conn)
scheme_df = pd.read_sql("select * from scheme_master",conn)
transaction_df = pd.read_sql("select * from investor_transactions",conn)

print(nav_df.head())
print(scheme_df.head())
print(transaction_df.head())

DatabaseError: Execution failed on sql 'select * from nav_history': no such table: nav_history

In [5]:
tables = pd.read_sql(""select name from sqlite_master where type='table';"",conn)

print(tables)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (3289278997.py, line 1)

In [6]:
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

print(tables)

                name
0           fact_nav
1  fact_transactions


In [7]:
nav_df = pd.read_sql("SELECT * FROM fact_nav", conn)
transaction_df = pd.read_sql("SELECT * FROM fact_transactions", conn)

print(nav_df.head())
print(nav_df.columns)

print(transaction_df.head())
print(transaction_df.columns)

   amfi_code        date       nav
0     100016  2022-01-03  520.4608
1     100016  2022-01-04  515.0971
2     100016  2022-01-05  521.7239
3     100016  2022-01-06  515.7880
4     100016  2022-01-07  515.1639
Index(['amfi_code', 'date', 'nav'], dtype='object')
  investor_id transaction_date  amfi_code transaction_type  amount_inr  \
0   INV003054       2024-01-01     119092              SIP        1834   
1   INV002952       2024-01-01     148567       Redemption      392882   
2   INV003420       2024-01-01     118636              SIP         912   
3   INV003436       2024-01-01     118634              SIP        1102   
4   INV004691       2024-01-01     119094          Lumpsum        8682   

         state       city city_tier age_group  gender  annual_income_lakh  \
0    Telangana  Hyderabad       T30       56+  Female                77.1   
1       Punjab   Amritsar       B30     18-25    Male                 7.1   
2      Haryana  Faridabad       B30     36-45    Male         

In [8]:
# Convert date column to datetime
nav_df["date"] = pd.to_datetime(nav_df["date"])

# Sort values
nav_df = nav_df.sort_values(["amfi_code", "date"])

# Daily Return
nav_df["daily_return"] = nav_df.groupby("amfi_code")["nav"].pct_change()

# Check
print(nav_df[["amfi_code", "date", "nav", "daily_return"]].head(10))

   amfi_code       date       nav  daily_return
0     100016 2022-01-03  520.4608           NaN
1     100016 2022-01-04  515.0971     -0.010306
2     100016 2022-01-05  521.7239      0.012865
3     100016 2022-01-06  515.7880     -0.011377
4     100016 2022-01-07  515.1639     -0.001210
5     100016 2022-01-10  510.7136     -0.008639
6     100016 2022-01-11  513.5542      0.005562
7     100016 2022-01-12  512.3195     -0.002404
8     100016 2022-01-13  510.2445     -0.004050
9     100016 2022-01-14  514.3636      0.008073


In [9]:
# Historical VaR (95%)
var_df = (
    nav_df.groupby("amfi_code")["daily_return"]
    .quantile(0.05)
    .reset_index()
)

var_df.columns = ["amfi_code", "VaR_95"]

print(var_df.head())

   amfi_code    VaR_95
0     100016 -0.014364
1     100025 -0.003793
2     100033 -0.019034
3     101206 -0.013282
4     101207 -0.026021


In [10]:
# Historical CVaR (95%)

def calculate_cvar(group):
    var = group["daily_return"].quantile(0.05)
    return group[group["daily_return"] <= var]["daily_return"].mean()

cvar_df = (
    nav_df.groupby("amfi_code")
    .apply(calculate_cvar)
    .reset_index(name="CVaR_95")
)

print(cvar_df.head())

   amfi_code   CVaR_95
0     100016 -0.018060
1     100025 -0.004994
2     100033 -0.023456
3     101206 -0.017439
4     101207 -0.032459


C:\Users\HP\AppData\Local\Temp\ipykernel_25380\3486823986.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calculate_cvar)


In [11]:
import numpy as np

# Daily risk-free rate (6.5% annual)
rf_daily = 0.065 / 252

nav_df["excess_return"] = nav_df["daily_return"] - rf_daily

rolling_sharpe = (
    nav_df.groupby("amfi_code")["excess_return"]
    .rolling(window=90)
    .apply(lambda x: (x.mean() / x.std()) * np.sqrt(252) if x.std() != 0 else np.nan)
    .reset_index()
)

rolling_sharpe.columns = ["amfi_code", "index", "rolling_sharpe"]

print(rolling_sharpe.head())

   amfi_code  index  rolling_sharpe
0     100016      0             NaN
1     100016      1             NaN
2     100016      2             NaN
3     100016      3             NaN
4     100016      4             NaN


In [12]:
# Investor Cohort Analysis

cohort_df = (
    transaction_df.groupby("age_group")
    .agg(
        investors=("investor_id", "nunique"),
        total_amount=("amount_inr", "sum"),
        avg_amount=("amount_inr", "mean")
    )
    .reset_index()
)

print(cohort_df)

  age_group  investors  total_amount     avg_amount
0     18-25        753     531639392  108144.709520
1     26-35       2033    1451600218  107821.452722
2     36-45       1237     871647528  107003.133808
3     46-55        589     405406469  107278.769251
4       56+        388     261286823  105613.105497


In [13]:
# SIP Continuity Analysis

sip_df = transaction_df[
    transaction_df["transaction_type"] == "SIP"
].copy()

sip_count = (
    sip_df.groupby("investor_id")
    .size()
    .reset_index(name="sip_transactions")
)

print(sip_count.head())

  investor_id  sip_transactions
0   INV000001                 2
1   INV000002                 3
2   INV000003                 2
3   INV000004                 6
4   INV000005                 3


In [14]:
# Merge VaR and CVaR reports

report_df = var_df.merge(cvar_df, on="amfi_code")

report_df.to_csv("var_cvar_report.csv", index=False)

print(report_df.head())

   amfi_code    VaR_95   CVaR_95
0     100016 -0.014364 -0.018060
1     100025 -0.003793 -0.004994
2     100033 -0.019034 -0.023456
3     101206 -0.013282 -0.017439
4     101207 -0.026021 -0.032459
